# Movie Recommendation System Using TensorFlow

This notebook is heavily inspiration from the [TensorFlow recommender basics tutorial series](https://www.tensorflow.org/recommenders/examples/basic_retrieval).

Real-world recommender systems are often composed of two stages:
- Retrival Stage
- Ranking Stage

## Dataset

In this notebook, we are going to use the [MovieLens dataset by GroupLens](https://www.kaggle.com/datasets/grouplens/movielens-20m-dataset). For a comprehensive overview, you can refer to the dataset page on Kaggle.

# Imports

In [1]:
!pip install tensorflow==2.15.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 475.3/475.3 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 66.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 91.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.0/442.0 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 5.1 MB/s eta 0:00:00
  Attempting uninstall: wrapt
    Found existing installation: wrapt 1.17.2
    Uninstalling wrapt-1.17.2:
      Successfully uninstalled wrapt-1.17.2
  Attempting uninstall: keras
    Found existing installation: keras 3.8.0
    Uninstalling keras-3.8.0:
      Successfully uninstalled keras-3.8.0
  Attempting uninstall: tensorboard
    Found existing installation: tensorboard 2.18.0
    Uninstalling tensorboard-2.18.0:
      Successfully uninstalled tensorboard-2.18.0
  Attempting uninstall: ml-dtypes
    Found existing ins

In [2]:
!pip install -q tensorflow-recommenders
!pip install -q --upgrade tensorflow-datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.2/96.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.9/319.9 kB 19.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.15.1 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.20.3, but you have protobuf 5.29.5 which is incompatible.
google-api-core 1.34.1 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<4.0.0dev,>=3.19.5, but you have protobuf 5.29.5 which is incompatible.
google-cloud-translate 3.12.1 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.19.5, but you have protobuf 5.29.5 which is incompatible.
tensorflow-decision-forests 1.11.0 requires tensorflo

In [3]:
import os
import pprint
import tempfile

from typing import Dict, Text

import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds

2025-06-01 05:06:19.478162: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-06-01 05:06:19.478233: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-06-01 05:06:19.480098: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [4]:
tf.__version__

'2.15.1'

In [5]:
import tensorflow_recommenders as tfrs

# EDA And Preprocessing

In [6]:
# Ratings data.
ratings = tfds.load("movielens/100k-ratings", split="train")
# Features of all the available movies.
movies = tfds.load("movielens/100k-movies", split="train")

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

...

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/movielens/100k-movies/incomplete.4XJ3Z0_0.1.1/movielens-train.tfrecord*...…

Dataset movielens downloaded and prepared to /root/tensorflow_datasets/movielens/100k-movies/0.1.1. Subsequent calls will reuse this data.


Check the values

In [7]:
for x in ratings.take(1).as_numpy_iterator():
  pprint.pprint(x)

{'bucketized_user_age': 45.0,
 'movie_genres': array([7]),
 'movie_id': b'357',
 'movie_title': b"One Flew Over the Cuckoo's Nest (1975)",
 'raw_user_age': 46.0,
 'timestamp': 879024327,
 'user_gender': True,
 'user_id': b'138',
 'user_occupation_label': 4,
 'user_occupation_text': b'doctor',
 'user_rating': 4.0,
 'user_zip_code': b'53211'}


In [8]:
for x in movies.take(1).as_numpy_iterator():
  pprint.pprint(x)

{'movie_genres': array([4]),
 'movie_id': b'1681',
 'movie_title': b'You So Crazy (1994)'}


## Feature Selection
We are only intersted in the user preference

In [9]:
ratings = ratings.map(lambda x: {
    "movie_title": x["movie_title"],
    "user_id": x["user_id"],
})
movies = movies.map(lambda x: x["movie_title"])

Print data after feature selection

In [10]:
for x in ratings.take(1).as_numpy_iterator():
  pprint.pprint(x)

{'movie_title': b"One Flew Over the Cuckoo's Nest (1975)", 'user_id': b'138'}


In [11]:
for x in movies.take(1).as_numpy_iterator():
  pprint.pprint(x)

b'You So Crazy (1994)'


## Train & Test Split

In [12]:
tf.random.set_seed(42)
train = ratings.shuffle(100_000, seed=42, reshuffle_each_iteration=False)

## Number of unique users and movies

In [13]:
movie_titles = movies.batch(1_000)
user_ids = ratings.batch(1_000_000).map(lambda x: x["user_id"])

unique_movie_titles = np.unique(np.concatenate(list(movie_titles)))
unique_user_ids = np.unique(np.concatenate(list(user_ids)))

unique_movie_titles[:10]

array([b"'Til There Was You (1997)", b'1-900 (1994)',
       b'101 Dalmatians (1996)', b'12 Angry Men (1957)', b'187 (1997)',
       b'2 Days in the Valley (1996)',
       b'20,000 Leagues Under the Sea (1954)',
       b'2001: A Space Odyssey (1968)',
       b'3 Ninjas: High Noon At Mega Mountain (1998)',
       b'39 Steps, The (1935)'], dtype=object)

# Two-tower Retrieval Model

Retrieval models are often composed of two sub-models:

1. A query model computing the query representation (normally a fixed-dimensionality embedding vector) using query features.
2. A candidate model computing the candidate representation (an equally-sized vector) using the candidate features

The outputs of the two models are then multiplied together to give a query-candidate affinity score, with higher scores expressing a better match between the candidate and the query.

In [14]:
embedding_dimension = 32

## Query Sub-Model

In [15]:
user_model = tf.keras.Sequential([
  tf.keras.layers.StringLookup(
      vocabulary=unique_user_ids, mask_token=None),
  # We add an additional embedding to account for unknown tokens.
  tf.keras.layers.Embedding(len(unique_user_ids) + 1, embedding_dimension)
])

## Candidate Sub-Model

In [16]:
movie_model = tf.keras.Sequential([
  tf.keras.layers.StringLookup(
      vocabulary=unique_movie_titles, mask_token=None),
  tf.keras.layers.Embedding(len(unique_movie_titles) + 1, embedding_dimension)
])

## Metrics

In [17]:
# Preformance
metrics = tfrs.metrics.FactorizedTopK(
  candidates=movies.batch(128).map(movie_model)
)

In [18]:
# Loss
task = tfrs.tasks.Retrieval(
  metrics=metrics
)

## Architecture

In [19]:
class MovielensModel(tfrs.Model):
  def __init__(self, user_model, movie_model):
    super().__init__()
    self.movie_model: tf.keras.Model = movie_model
    self.user_model: tf.keras.Model = user_model
    self.task: tf.keras.layers.Layer = task

  def compute_loss(self, features: Dict[Text, tf.Tensor], training=False) -> tf.Tensor:
    # We pick out the user features and pass them into the user model.
    user_embeddings = self.user_model(features["user_id"])
    # And pick out the movie features and pass them into the movie model,
    # getting embeddings back.
    positive_movie_embeddings = self.movie_model(features["movie_title"])

    # The task computes the loss and the metrics.
    return self.task(user_embeddings, positive_movie_embeddings)

In [20]:
retrival_model = MovielensModel(user_model, movie_model)
retrival_model.compile(optimizer=tf.keras.optimizers.Adagrad(learning_rate=0.1))

In [21]:
cached_train = train.shuffle(100_000).batch(8192).cache()

In [22]:
retrival_model.fit(cached_train, epochs=3)

Epoch 1/3
13/13 [==============================] - 24s 2s/step - factorized_top_k/top_1_categorical_accuracy: 0.0016 - factorized_top_k/top_5_categorical_accuracy: 0.0117 - factorized_top_k/top_10_categorical_accuracy: 0.0242 - factorized_top_k/top_50_categorical_accuracy: 0.1160 - factorized_top_k/top_100_categorical_accuracy: 0.2022 - loss: 64357.4643 - regularization_loss: 0.0000e+00 - total_loss: 64357.4643
Epoch 2/3
13/13 [==============================] - 17s 1s/step - factorized_top_k/top_1_categorical_accuracy: 0.0028 - factorized_top_k/top_5_categorical_accuracy: 0.0189 - factorized_top_k/top_10_categorical_accuracy: 0.0383 - factorized_top_k/top_50_categorical_accuracy: 0.1699 - factorized_top_k/top_100_categorical_accuracy: 0.2933 - loss: 62056.9932 - regularization_loss: 0.0000e+00 - total_loss: 62056.9932
Epoch 3/3
13/13 [==============================] - 19s 1s/step - factorized_top_k/top_1_categorical_accuracy: 0.0032 - factorized_top_k/top_5_categorical_accuracy: 0.0204

In [23]:
retrival_model.evaluate(cached_train, return_dict=True)

13/13 [==============================] - 17s 1s/step - factorized_top_k/top_1_categorical_accuracy: 0.0048 - factorized_top_k/top_5_categorical_accuracy: 0.0281 - factorized_top_k/top_10_categorical_accuracy: 0.0551 - factorized_top_k/top_50_categorical_accuracy: 0.2092 - factorized_top_k/top_100_categorical_accuracy: 0.3406 - loss: 60502.0131 - regularization_loss: 0.0000e+00 - total_loss: 60502.0131


{'factorized_top_k/top_1_categorical_accuracy': 0.004819999914616346,
 'factorized_top_k/top_5_categorical_accuracy': 0.028130000457167625,
 'factorized_top_k/top_10_categorical_accuracy': 0.05511000007390976,
 'factorized_top_k/top_50_categorical_accuracy': 0.2091899961233139,
 'factorized_top_k/top_100_categorical_accuracy': 0.3406499922275543,
 'loss': 11387.822265625,
 'regularization_loss': 0,
 'total_loss': 11387.822265625}

## Inference

In [24]:
# Create a model that takes in raw query features, and
index = tfrs.layers.factorized_top_k.BruteForce(retrival_model.user_model)
# recommends movies out of the entire movies dataset.
index.index_from_dataset(
  tf.data.Dataset.zip((movies.batch(100), movies.batch(100).map(retrival_model.movie_model)))
)

# Get recommendations.
_, titles = index(tf.constant(["42"]))
print(f"Recommendations for user 42: {titles[0, :3]}")

Recommendations for user 42: [b'Rudy (1993)' b'Murder in the First (1995)' b'Client, The (1994)']


# Ranking Model

## Data Preparation

In [25]:
ratings = tfds.load("movielens/100k-ratings", split="train")

ratings = ratings.map(lambda x: {
    "movie_title": x["movie_title"],
    "user_id": x["user_id"],
    "user_rating": x["user_rating"]
})

In [26]:
tf.random.set_seed(42)
shuffled = ratings.shuffle(100_000, seed=42, reshuffle_each_iteration=False)

train = shuffled.take(80_000)
test = shuffled.skip(80_000).take(20_000)

In [27]:
movie_titles = ratings.batch(1_000_000).map(lambda x: x["movie_title"])
user_ids = ratings.batch(1_000_000).map(lambda x: x["user_id"])

unique_movie_titles = np.unique(np.concatenate(list(movie_titles)))
unique_user_ids = np.unique(np.concatenate(list(user_ids)))

## Architecture

In [28]:
class RankingModel(tf.keras.Model):

  def __init__(self):
    super().__init__()
    embedding_dimension = 32

    # Compute embeddings for users.
    self.user_embeddings = tf.keras.Sequential([
      tf.keras.layers.StringLookup(
        vocabulary=unique_user_ids, mask_token=None),
      tf.keras.layers.Embedding(len(unique_user_ids) + 1, embedding_dimension)
    ])

    # Compute embeddings for movies.
    self.movie_embeddings = tf.keras.Sequential([
      tf.keras.layers.StringLookup(
        vocabulary=unique_movie_titles, mask_token=None),
      tf.keras.layers.Embedding(len(unique_movie_titles) + 1, embedding_dimension)
    ])

    # Compute predictions.
    self.ratings = tf.keras.Sequential([
      # Learn multiple dense layers.
      tf.keras.layers.Dense(256, activation="relu"),
      tf.keras.layers.Dense(64, activation="relu"),
      # Make rating predictions in the final layer.
      tf.keras.layers.Dense(1)
    ])

  def call(self, inputs):

    user_id, movie_title = inputs

    user_embedding = self.user_embeddings(user_id)
    movie_embedding = self.movie_embeddings(movie_title)

    return self.ratings(tf.concat([user_embedding, movie_embedding], axis=1))

In [29]:
# This model takes user ids and movie titles, and outputs a predicted rating
RankingModel()((["42"], ["One Flew Over the Cuckoo's Nest (1975)"]))

<tf.Tensor: shape=(1, 1), dtype=float32, numpy=array([[0.02406374]], dtype=float32)>

## Loss and metrics

In [30]:
rating_task = tfrs.tasks.Ranking(
  loss = tf.keras.losses.MeanSquaredError(),
  metrics=[tf.keras.metrics.RootMeanSquaredError()]
)

## Full Model Implementation

In [31]:
class MovielensModel(tfrs.models.Model):

  def __init__(self):
    super().__init__()
    self.ranking_model: tf.keras.Model = RankingModel()
    self.task: tf.keras.layers.Layer = rating_task

  def call(self, features: Dict[str, tf.Tensor]) -> tf.Tensor:
    return self.ranking_model(
        (features["user_id"], features["movie_title"]))

  def compute_loss(self, features: Dict[Text, tf.Tensor], training=False) -> tf.Tensor:
    labels = features.pop("user_rating")

    rating_predictions = self(features)

    # The task computes the loss and the metrics.
    return self.task(labels=labels, predictions=rating_predictions)

In [32]:
ranking_model = MovielensModel()
ranking_model.compile(optimizer=tf.keras.optimizers.Adagrad(learning_rate=0.1))

In [33]:
cached_train = train.shuffle(100_000).batch(8192).cache()
cached_test = test.batch(4096).cache()

In [34]:
ranking_model.fit(cached_train, epochs=3)

Epoch 1/3
10/10 [==============================] - 3s 29ms/step - root_mean_squared_error: 2.1463 - loss: 4.2361 - regularization_loss: 0.0000e+00 - total_loss: 4.2361
Epoch 2/3
10/10 [==============================] - 0s 18ms/step - root_mean_squared_error: 1.1204 - loss: 1.2573 - regularization_loss: 0.0000e+00 - total_loss: 1.2573
Epoch 3/3
10/10 [==============================] - 0s 18ms/step - root_mean_squared_error: 1.1108 - loss: 1.2357 - regularization_loss: 0.0000e+00 - total_loss: 1.2357


In [35]:
ranking_model.evaluate(cached_test, return_dict=True)

5/5 [==============================] - 2s 13ms/step - root_mean_squared_error: 1.1037 - loss: 1.2138 - regularization_loss: 0.0000e+00 - total_loss: 1.2138


{'root_mean_squared_error': 1.103699803352356,
 'loss': 1.1946895122528076,
 'regularization_loss': 0,
 'total_loss': 1.1946895122528076}

## Inference

In [36]:
test_ratings = {}
test_movie_titles = ["M*A*S*H (1970)", "Dances with Wolves (1990)", "Speed (1994)"]
for movie_title in test_movie_titles:
  test_ratings[movie_title] = ranking_model({
      "user_id": np.array(["42"]),
      "movie_title": np.array([movie_title])
  })

print("Ratings:")
for title, score in sorted(test_ratings.items(), key=lambda x: x[1], reverse=True):
  print(f"{title}: {score}")

Ratings:
Dances with Wolves (1990): [[3.7193525]]
M*A*S*H (1970): [[3.660066]]
Speed (1994): [[3.6569092]]


# Full Recommder System

Combine the retrival model with the ranking model to provide accurate recommendations.

In [37]:
class Recommender():
    def __init__(self, retrival_model, ranking_model):
        index = tfrs.layers.factorized_top_k.BruteForce(retrival_model.user_model)
        index.index_from_dataset(
          tf.data.Dataset.zip((movies.batch(100), movies.batch(100).map(retrival_model.movie_model)))
        )
        self.retrival_model = index
        self.ranking_model = ranking_model
    
    def predict(self, user_id):
        _, titles = self.retrival_model(tf.constant([user_id]))
        ratings = []
        for index, title in enumerate(titles[0]):
            ratings.append(self.ranking_model({
                "user_id": np.array([user_id]),
                "movie_title": np.array([title.numpy().decode('utf-8')])
            }))
        return sorted(zip(titles.numpy().tolist()[0], tf.squeeze(tf.concat(ratings, axis=0)).numpy().tolist()), key=lambda x: x[1], reverse=True)

In [38]:
recommender_model = Recommender(retrival_model, ranking_model)

In [39]:
recommender_model.predict("12")

[(b'Forrest Gump (1994)', 3.7515063285827637),
 (b'Dead Poets Society (1989)', 3.746403217315674),
 (b'Dances with Wolves (1990)', 3.7380995750427246),
 (b'Fried Green Tomatoes (1991)', 3.7229859828948975),
 (b"Schindler's List (1993)", 3.7214128971099854),
 (b'Field of Dreams (1989)', 3.7147834300994873),
 (b'E.T. the Extra-Terrestrial (1982)', 3.6952924728393555),
 (b'Apollo 13 (1995)', 3.6921443939208984),
 (b'Glory (1989)', 3.6654176712036133),
 (b'Little Women (1994)', 3.658749580383301)]